In [1]:
#------------------------------------------------ Import Lib ----------------------------------------
import os
import re
import datetime
import requests
import pandas as pd
from bs4 import BeautifulSoup
from time import sleep

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'BN AMBD' ## Monetary Authority of Brunei Darussalam (now Brunei Darussalam Central Bank, BDCB)

print(f"Running {regulatorName} Web Scraping Tool v.1.0")
now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

#scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## for the production environment
except NameError:
    scriptfolder = f"/Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/{regulatorName}"

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running BN AMBD Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}

# All 6 lists come from the same BDCB licensee register, filtered by license type
# (Jira DECD-5885). Filters are passed as GET params: flt_sector[] = whole sector,
# flt_subs[] = individual license types. view=100 keeps every list on a single page.
BASE = 'https://www.bdcb.gov.bn/regulatory/list-of-bdcb-licensees'

regdict={

        # 1 - Banks and Finance Companies: whole sector
        regulatorName + ' 1': BASE + '?view=100&flt_sector%5B%5D=01hhgca0vwscpervey9gqbt7s6',
        # 2 - Takaful and Insurance: skip Termination/Suspension and Individual Agents
        regulatorName + ' 2': BASE + '?view=100'
                              '&flt_subs%5B%5D=01hhgd4ghsn1bm4r37cwyqcwa2'   # Insurance Companies - Life
                              '&flt_subs%5B%5D=01hhgd4s89g48mp2m71rvzze61'   # Insurance Companies - Non-life
                              '&flt_subs%5B%5D=01hhgd5fbs5tz197f0s1eb8k4k'   # Takaful Operators - General
                              '&flt_subs%5B%5D=01hhgd662g5n05v295ctd9edg3'   # Takaful Operators - Family
                              '&flt_subs%5B%5D=01hhgd6xse4etnjnvgvtwcvzvk'   # Corporate Agents
                              '&flt_subs%5B%5D=01hq35thsyvhhcyc1pjnhszmnq'   # Brokers
                              '&flt_subs%5B%5D=01k2nnx7av1k11ecjq5tw658h2',  # Takaful Agents
        # 3 - Capital Market: just CMSL and CISL, skip ceased filters
        regulatorName + ' 3': BASE + '?view=100'
                              '&flt_subs%5B%5D=01hhgd03wyn341rphya5h08vw0'   # CMSL
                              '&flt_subs%5B%5D=01hhgd134nn9m3er52ycaavrvm',  # CISL
        # 4 - Payment System Operators: whole sector
        regulatorName + ' 4': BASE + '?view=100&flt_sector%5B%5D=01hhgcb8bs0vhpn164zrmc53v2',
        # 5 - Specialised Market (Money Changers / Money Remitters / Pawnbrokers): whole sector
        regulatorName + ' 5': BASE + '?view=100&flt_sector%5B%5D=01hhgcbg2pqfcchprmvj25jg84',
        # 6 - Others (AML/CFT, Supervised by BDCB): whole sector
        regulatorName + ' 6': BASE + '?view=100&flt_sector%5B%5D=01hsk82fecntyzshx2nec8va5t',

        }


Typology={

       regulatorName + ' 1': 'Banks and Finance Companies',
       regulatorName + ' 2': 'Takaful and Insurance',
       regulatorName + ' 3': 'Capital Market',
       regulatorName + ' 4': 'Payment System Operators',
       regulatorName + ' 5': 'Specialised Market',
       regulatorName + ' 6': 'Others',

        }

processdate = now.strftime('%Y-%m-%d')

In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


HEADERS = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36'}

def fetch(url, max_retry=4):
    # the site drops connections when hit in quick bursts -> pace + retry with backoff
    for attempt in range(max_retry):
        try:
            r = requests.get(url, headers=HEADERS, timeout=60)
            if r.status_code == 200:
                return r.text
            print(f'[WARN] : HTTP {r.status_code}, retry {attempt+1}/{max_retry}')
        except requests.RequestException as e:
            print(f'[WARN] : {e.__class__.__name__}, retry {attempt+1}/{max_retry}')
        sleep(30)
    raise RuntimeError(f'failed to fetch {url}')


def parse_cards(html):
    # each licensee card: <p class="h5default">Name</p> + tagline "License Type | Sector"
    # + accordion body with labelled blocks (<p class="font-bold text-green">Label</p><p>value</p>)
    soup = BeautifulSoup(html, 'html.parser')
    cards = []
    for name_p in soup.select('p.h5default'):
        container = name_p.parent
        tag_p = container.select_one('p.font-secondary')
        licence = ''
        if tag_p:
            licence = ' '.join(tag_p.get_text(' ', strip=True).split('|')[0].split())
        fields = {}
        for lab in container.select('p.font-bold.text-green'):
            vals = []
            sib = lab.find_next_sibling()
            while sib is not None and not (sib.name == 'p' and 'font-bold' in (sib.get('class') or [])):
                t = sib.get_text('\n', strip=True)
                t = '\n'.join(' '.join(line.split()) for line in t.split('\n') if line.strip())  # kill stray \r / double spaces
                if t:
                    vals.append(t)
                sib = sib.find_next_sibling()
            fields[lab.get_text(strip=True)] = '\n'.join(vals)
        cards.append({'name': ' '.join(name_p.get_text(strip=True).split()), 'licence': licence, 'fields': fields})
    return cards


def parse_contacts(text):
    tel = fax = email = website = ''
    for line in text.split('\n'):
        s = line.strip()
        if not s:
            continue
        m = re.search(r'(?:tel|telephone|phone)\s*(?:no\.?|number)?\s*[:.]?\s*(.+)', s, re.I)
        if m and not tel:
            tel = m.group(1).strip()
            continue
        m = re.search(r'fax\s*[:.]?\s*(.+)', s, re.I)
        if m and not fax:
            fax = m.group(1).strip()
            continue
        m = re.search(r'e-?mail\s*[:.]?\s*(.+)', s, re.I)
        if m and not email:
            email = m.group(1).strip()
            continue
        m = re.search(r'web\s*site|web\s*[:.]', s, re.I)
        if m and not website:
            website = re.sub(r'^\s*web\s*site?\s*[:.]?\s*', '', s, flags=re.I).strip()
            continue
        if re.search(r'https?://|www\.', s, re.I) and not website:
            website = s
        elif '@' in s and not email:
            email = s
    return tel, fax, email, website

In [5]:
#------------------------------------------------ Begin_Main ----------------------------------------
# NOTE: termination is handled by NOT selecting the Termination/Suspension filter (per Jira);
# entities the register still shows under the selected filters are kept even if their card
# carries a Status note (confirmed with user, e.g. Halalan Toyyiban Insurance & Takaful Agency).

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    rows_before = len(sqldict['Name'])
    page = 1
    while True:
        html = fetch(regdict[reg] + f'&page={page}')
        cards = parse_cards(html)
        if not cards:
            break
        for card in cards:
            address = card['fields'].get('Address') or card['fields'].get('Business Address') or ''
            tel, fax, email, website = parse_contacts(card['fields'].get('Contact Details', ''))
            sqldict['Name'].append(card['name'])
            sqldict['License_Type'].append(card['licence'])
            sqldict['Address_1'].append(address.replace('\n', ' ').strip())
            sqldict['Phone'].append(tel)
            sqldict['Fax'].append(fax)
            sqldict['Email'].append(email)
            sqldict['Website'].append(website)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['RegulationType'].append('Regulated')
            sqldict = bourange_same_length_array(sqldict)
        if len(cards) < 100:  # view=100 -> a full page means there may be another one
            break
        page += 1
        sleep(12)
    print(f"[INFO] : {reg} collected {len(sqldict['Name']) - rows_before} rows")
    sleep(12)

[INFO] : Working 1/6 _(BN AMBD 1)_ 


[INFO] : BN AMBD 1 collected 10 rows


[INFO] : Working 2/6 _(BN AMBD 2)_ 


[INFO] : BN AMBD 2 collected 62 rows


[INFO] : Working 3/6 _(BN AMBD 3)_ 


...

[INFO] : BN AMBD 4 collected 6 rows


[INFO] : Working 5/6 _(BN AMBD 5)_ 


[INFO] : BN AMBD 5 collected 40 rows


[INFO] : Working 6/6 _(BN AMBD 6)_ 


[INFO] : BN AMBD 6 collected 2 rows


In [6]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df = df[df['Name'] != '']
outfile = os.path.join(tempfolder, filename)
df.to_excel(outfile, sheet_name='SQL Ready', index=False)
print(f'[INFO] : saved {len(df)} rows -> {outfile}')

[INFO] : saved 157 rows -> /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/BN AMBD/tempfolder/BN AMBD SQL Ready 2026-07-02 20.23.33.xlsx
